In [ ]:
class SNRCalc():
    def __init__(self, dt, df, t1_myelin, t2_myelin, t1_wm, t2_wm, img_size = 256, n_readout = 9, flip_angle = math.pi/2):
        self.dt = dt
        self.df = df
        self.t1_myelin = t1_myelin
        self.t2_myelin = t2_myelin
        self.t1_wm = t1_wm
        self.t2_wm = t2_wm
        self.img_size = img_size
        self.n_readout = n_readout
        self.flip_angle = flip_angle
        pass

    def combined_kdata_ktraj(self, t_readout, prob_map_myelin, prob_map_wm):
        print('white matter')
        wm = ute.IR_UTE(self.dt, self.df, self.t1_wm, self.t2_wm)
        _ = wm.inversion_recovery()
        t_nullout = wm.calculate_nulling_point()
        _, _, mag_readout_start_wm, _ = wm.transverse_excitation(self.n_readout, t_readout, t_nullout, self.flip_angle)

        print()
        print('myelin')
        myelin = ute.IR_UTE(self.dt, self.df, self.t1_myelin, self.t2_myelin)
        _ = myelin.inversion_recovery()
        _, _, mag_readout_start_myelin, _ = myelin.transverse_excitation(self.n_readout, t_readout, t_nullout, self.flip_angle)


        radial_sampling = rs.RadialSampling(self.img_size)
        self.nspokes = radial_sampling.nspokes
        kdata_t_myelin, ktraj_total = radial_sampling.get_kdata_ktraj(self.df, mag_readout_start_myelin, prob_map_myelin, self.n_readout, self.t2_myelin)
        kdata_t_wm, _ = radial_sampling.get_kdata_ktraj(self.df, mag_readout_start_wm, prob_map_wm, self.n_readout, self.t2_wm)
        
        kdata_total = kdata_t_myelin + kdata_t_wm
        self.radial_sampling = radial_sampling


        print("flip_angle:", self.flip_angle)
        print("mag_readout_start_myelin:", mag_readout_start_myelin)
        print("mag_readout_start_wm:", mag_readout_start_wm)
        return kdata_total, ktraj_total
    
    def SNR_CNR_calculation(self, kdata_total, ktraj_total, prob_map_myelin, prob_map_wm, blur=False):
        if blur:
            # method 1: no density compensation (blurry image)
            image = self.radial_sampling.adjnufft_ob(kdata_total, ktraj_total)
        else:
            # method 2: use density compensation
            dcomp = tkbn.calc_density_compensation_function(ktraj=ktraj_total, im_size=(self.img_size, self.img_size))
            image = self.radial_sampling.adjnufft_ob(kdata_total * dcomp, ktraj_total)

    
        image_numpy = np.squeeze(image.cpu().numpy())
        print(image_numpy.shape)
        image_abs = np.abs(image_numpy) 

        noise_std = np.std(image_abs[(prob_map_myelin == 0) & (prob_map_wm == 0)])
        print(image_abs[(prob_map_myelin == 0) & (prob_map_wm == 0)])
        myelin_mean = np.mean(image_abs[prob_map_myelin == 1])
        wm_mean = np.mean(image_abs[prob_map_wm == 1])

        print('noise_std:', noise_std)
        print('noise mean:', np.mean(image_abs[(prob_map_myelin == 0) & (prob_map_wm == 0)]))
        print('myelin_mean:', myelin_mean)
        print('wm_mean:', wm_mean)
        
        snr_myelin = 0.66 * myelin_mean / noise_std
        snr_wm = 0.66 * wm_mean / noise_std
        cnr = (snr_myelin - snr_wm) / noise_std
        print("SNR_myelin:", snr_myelin)
        print("SNR_wm:", snr_wm)
        print('SNR Noise:', 0.66 * np.mean(image_abs[(prob_map_myelin == 0) & (prob_map_wm == 0)])/noise_std)
        return  snr_myelin, snr_wm, cnr
    
    def total_acquisation_time_calc(self, TR):
        """
        TR: repetition time in ms
        """
        radial_sampling = rs.RadialSampling(self.img_size)
        nspokes = radial_sampling.nspokes

        n_TR = np.ceil(nspokes / self.n_readout)
        total_TA = n_TR * TR
        return total_TA

   



# combined = SNRCalc(dt, df, t1_myelin, t2_myelin, t1_wm, t2_wm, img_size = img_size, n_readout = n_readout, flip_angle = flip_angle)
# kdata_total, ktraj_total = combined.combined_kdata_ktraj(t_readout, prob_map_myelin, prob_map_wm)
# snr_myelin, snr_wm = combined.SNR_calculation(kdata_total, ktraj_total, prob_map_myelin, prob_map_wm)



## (1) Plot flip angle vs SNR across different n_readout

In [ ]:
def xxx(n_readout_list, angles_deg_list):
    flip_angle_list = [angle * math.pi / 180 for angle in angles_deg_list]
    for i in n_readout_list:
        snr_myelin_list, snr_wm_list = [], []
        for flip_angle in flip_angle_list:
            combined = SNRCalc(dt, df, t1_myelin, t2_myelin, t1_wm, t2_wm, img_size = img_size, n_readout = n_readout, flip_angle = flip_angle)
            kdata_total, ktraj_total = combined.combined_kdata_ktraj(t_readout, prob_map_myelin, prob_map_wm)
            snr_myelin, snr_wm = combined.SNR_CNR_calculation(kdata_total, ktraj_total, prob_map_myelin, prob_map_wm)
            snr_myelin_list.append(snr_myelin)
            snr_wm_list.append(snr_wm)
        plt.plot(angles_deg_list, snr_myelin_list)
        plt.plot(angles_deg_list, snr_wm_list)
        plt.title(f'spoke = {i}')
    pass


angles_deg_list = [10, 20, 30, 40, 50, 60, 70, 80, 90]
n_readout_list = [1,3,5,7,9] 

In [ ]:
import math
import matplotlib.pyplot as plt

def readout_angle_pairs_display(n_readout_list, angles_deg_list):
    flip_angle_list = [angle * torch.pi / 180 for angle in angles_deg_list]

    n_plots = len(n_readout_list)
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4), sharey=True)

    if n_plots == 1:
        axes = [axes]

    for ax, n_readout in zip(axes, n_readout_list):
        snr_myelin_list, snr_wm_list, cnr_list, total_TA_list = [], [], [], []

        for flip_angle in flip_angle_list:
            combined = SNRCalc(
                dt, df,
                t1_myelin, t2_myelin,
                t1_wm, t2_wm,
                img_size=img_size,
                n_readout=n_readout,
                flip_angle=flip_angle
            )

            kdata_total, ktraj_total = combined.combined_kdata_ktraj(
                t_readout, prob_map_myelin, prob_map_wm
            )

            snr_myelin, snr_wm, cnr = combined.SNR_CNR_calculation(
                kdata_total, ktraj_total, prob_map_myelin, prob_map_wm
            )

            total_TA = combined.total_acquisation_time_calc()

            snr_myelin_list.append(snr_myelin)
            snr_wm_list.append(snr_wm)
            cnr_list.append(cnr)
            total_TA_list.append(total_TA)


        ax.plot(angles_deg_list, snr_myelin_list, label='Myelin SNR')
        ax.plot(angles_deg_list, snr_wm_list, label='WM SNR')
        ax.set_title(f'n_readout = {n_readout}')
        ax.set_xlabel('Flip angle (deg)')
        ax.legend()
        ax.grid(True)

    axes[0].set_ylabel('SNR')

    plt.tight_layout()
    plt.show()

In [ ]:
import math
import torch
import numpy as np
import IR_UTE as ute
import phantom as pt
import torchkbnufft as tkbn
import randialSampling as rs
import matplotlib.pyplot as plt

class SNRCalc():
    def __init__(self, dt, df, t1_myelin, t2_myelin, t1_wm, t2_wm, img_size = 256, n_readout = 9, flip_angle = math.pi/2):
        self.dt = dt
        self.df = df
        self.t1_myelin = t1_myelin
        self.t2_myelin = t2_myelin
        self.t1_wm = t1_wm
        self.t2_wm = t2_wm
        self.img_size = img_size
        self.n_readout = n_readout
        self.flip_angle = flip_angle
        pass

    def combined_kdata_ktraj(self, t_readout, prob_map_myelin, prob_map_wm):
        print('white matter')
        wm = ute.IR_UTE(self.dt, self.df, self.t1_wm, self.t2_wm)
        _ = wm.inversion_recovery()
        t_nullout = wm.calculate_nulling_point()
        _, _, mag_readout_start_wm, _ = wm.transverse_excitation(self.n_readout, t_readout, t_nullout, self.flip_angle, True)

        print()
        print('myelin')
        myelin = ute.IR_UTE(self.dt, self.df, self.t1_myelin, self.t2_myelin)
        _ = myelin.inversion_recovery()
        _, _, mag_readout_start_myelin, _ = myelin.transverse_excitation(self.n_readout, t_readout, t_nullout, self.flip_angle)


        radial_sampling = rs.RadialSampling(self.img_size)
        kdata_t_myelin, ktraj_total = radial_sampling.get_kdata_ktraj(self.df, mag_readout_start_myelin, prob_map_myelin, self.n_readout, self.t2_myelin)
        kdata_t_wm, _ = radial_sampling.get_kdata_ktraj(self.df, mag_readout_start_wm, prob_map_wm, self.n_readout, self.t2_wm)
        
        kdata_total = kdata_t_myelin + kdata_t_wm
        self.radial_sampling = radial_sampling


        print("flip_angle:", self.flip_angle)
        print("mag_readout_start_myelin:", mag_readout_start_myelin)
        print("mag_readout_start_wm:", mag_readout_start_wm)
        return kdata_total, ktraj_total
    
    def SNR_calculation(self, kdata_total, ktraj_total, prob_map_myelin, prob_map_wm, blur=False):
        if blur:
            # method 1: no density compensation (blurry image)
            image = self.radial_sampling.adjnufft_ob(kdata_total, ktraj_total)
        else:
            # method 2: use density compensation
            dcomp = tkbn.calc_density_compensation_function(ktraj=ktraj_total, im_size=(self.img_size, self.img_size))
            image = self.radial_sampling.adjnufft_ob(kdata_total * dcomp, ktraj_total)
    
        image_numpy = np.squeeze(image.cpu().numpy())
        image_abs = np.abs(image_numpy) 

        noise_std = np.std(image_abs[(prob_map_myelin == 0) & (prob_map_wm == 0)])
        myelin_mean = np.mean(image_abs[prob_map_myelin == 1])
        wm_mean = np.mean(image_abs[prob_map_wm == 1])
        
        snr_myelin = 0.66 * myelin_mean / noise_std
        snr_wm = 0.66 * wm_mean / noise_std
        return  snr_myelin, snr_wm
    
    def total_acquisation_time_calc(self, TR):
        """
        TR: repetition time in ms
        """
        radial_sampling = rs.RadialSampling(self.img_size)
        nspokes = radial_sampling.nspokes

        n_TR = np.ceil(nspokes / self.n_readout)
        total_TA = n_TR * TR
        return total_TA



   


   



In [ ]:


n_on_resonance = 100
magnetization = full_wm1
magnetization_xy1 = np.sqrt(np.power(magnetization[n_on_resonance, :, 0], 2) + np.power(magnetization[n_on_resonance, :, 1], 2))
n_steps =  magnetization.shape[1]

time1 = np.arange(n_steps) * dt  # micro seconds
plt.figure(figsize=(6,4))
plt.plot(time1[:7000], magnetization_xy1[:7000], label= "test")

plt.xlabel('Time (us)')
plt.ylabel('Transverse Magnetisation Mxy')
#plt.ylim([0,1])
plt.title(f'{30} Degrees Exciation at Resonance (Sinc Pulse)')
plt.grid(True)
plt.legend(title = "Group")
plt.show()

### Phantom class (Old version)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class VirtualPhantom():
    def __init__(self, img_size):
        self.img_size = img_size
        self.inner_size = int(0.25* img_size)
        self.mixed_size = int(0.50 * img_size)
        self.outer_size = int(0.75 * img_size)
        self.upper_square = int(0.078 * img_size)

        # gap between upper square and mixed square
        self.gap = max(1, int(0.02 * img_size))

        # Inner square (pure myelin) - centered
        self.top_inner = (self.img_size - self.inner_size) // 2
        self.bottom_inner = self.top_inner + self.inner_size

        # Mixed square (mixed signals) - centered
        self.top_mixed = (self.img_size - self.mixed_size) // 2
        self.bottom_mixed = self.top_mixed + self.mixed_size

        # Outer square (pure WM) - centered
        self.top_outer = (self.img_size - self.outer_size) // 2
        self.bottom_outer = self.top_outer + self.outer_size

        # Upper square: centered horizontally, above mixed square, with a gap
        self.x0 = self.img_size // 2 - self.upper_square // 2
        self.x1 = self.x0 + self.upper_square

        self.y1 = self.top_mixed - self.gap
        self.y0 = self.y1 - self.upper_square

        if self.y0 < 0:
            self.y0 = 0
            self.y1 = self.upper_square

    def create_maps(self, t1, t2, inner, mixed, outer, upper_sqaure=True):
        # T1 map
        self.T1_map = np.full([self.img_size, self.img_size], t1, dtype=np.float64)

        # T2 map
        self.T2_map = np.full([self.img_size, self.img_size], t2, dtype=np.float64)

        # Probability map
        prob_map = np.zeros([self.img_size, self.img_size], dtype=np.float64)
        prob_map[self.top_outer:self.bottom_outer, self.top_outer:self.bottom_outer] = outer
        prob_map[self.top_mixed:self.bottom_mixed, self.top_mixed:self.bottom_mixed] = mixed
        prob_map[self.top_inner:self.bottom_inner, self.top_inner:self.bottom_inner] = inner
        if upper_sqaure:
            prob_map[self.y0:self.y1, self.x0:self.x1] = 1

        self.prob_map = prob_map
        return self.T1_map, self.T2_map, self.prob_map

    def phantom_display(self, title):
        plt.figure(figsize=(5, 5))

        plt.subplot(2, 2, 1)
        plt.imshow(self.T1_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} T1 Map")

        plt.subplot(2, 2, 2)
        plt.imshow(self.T2_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} T2 Map")

        plt.subplot(2, 2, 3)
        plt.imshow(self.prob_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} Probability Map")

        plt.tight_layout()
        plt.show()



## Phantom class (version 2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class VirtualPhantom():
    def __init__(self, img_size):
        self.img_size = img_size
        self.inner_size = int(0.25* img_size)
        self.mixed_size = int(0.50 * img_size)
        self.outer_size = int(0.75 * img_size)
        self.upper_square = int(0.078 * img_size)

        # gap between upper square and mixed square
        self.gap = max(1, int(0.02 * img_size))

        # Inner square (pure myelin) - centered
        self.top_inner = (self.img_size - self.inner_size) // 2
        self.bottom_inner = self.top_inner + self.inner_size

        # Mixed square (mixed signals) - centered
        self.top_mixed = (self.img_size - self.mixed_size) // 2
        self.bottom_mixed = self.top_mixed + self.mixed_size

        # Outer square (pure WM) - centered
        self.top_outer = (self.img_size - self.outer_size) // 2
        self.bottom_outer = self.top_outer + self.outer_size

        # Upper square: centered horizontally, above mixed square, with a gap
        self.x0 = self.img_size // 2 - self.upper_square // 2
        self.x1 = self.x0 + self.upper_square

        self.y1 = self.top_mixed - self.gap
        self.y0 = self.y1 - self.upper_square

        if self.y0 < 0:
            self.y0 = 0
            self.y1 = self.upper_square


    def create_artefact_ROI_region(self, width):
        artefact_ROI_region = np.zeros([self.img_size, self.img_size], dtype = np.float64)
        artefact_ROI_region[self.top_inner-width : self.top_inner+width, 0:self.top_outer] = 1
        artefact_ROI_region[self.bottom_inner-width : self.bottom_inner+width, 0:self.top_outer] = 1

        artefact_ROI_region[self.top_inner-width : self.top_inner+width, self.img_size - self.top_outer: self.img_size] = 1
        artefact_ROI_region[self.bottom_inner-width : self.bottom_inner+width, self.img_size - self.top_outer: self.img_size] = 1

        artefact_ROI_region[0:self.top_outer, self.top_inner-width : self.top_inner+width] = 1
        artefact_ROI_region[0:self.top_outer, self.bottom_inner-width : self.bottom_inner+width] = 1

        artefact_ROI_region[self.img_size - self.top_outer: self.img_size, self.top_inner-width : self.top_inner+width] = 1
        artefact_ROI_region[self.img_size - self.top_outer: self.img_size, self.bottom_inner-width : self.bottom_inner+width] = 1

        return artefact_ROI_region


    def create_maps(self, t1, t2, inner, mixed, outer, upper_sqaure=True):
        # T1 map
        self.T1_map = np.full([self.img_size, self.img_size], t1, dtype=np.float64)

        # T2 map
        self.T2_map = np.full([self.img_size, self.img_size], t2, dtype=np.float64)

        # Probability map
        prob_map = np.zeros([self.img_size, self.img_size], dtype=np.float64)
        prob_map[self.top_outer:self.bottom_outer, self.top_outer:self.bottom_outer] = outer
        prob_map[self.top_mixed:self.bottom_mixed, self.top_mixed:self.bottom_mixed] = mixed
        prob_map[self.top_inner:self.bottom_inner, self.top_inner:self.bottom_inner] = inner
        if upper_sqaure:
            prob_map[self.y0:self.y1, self.x0:self.x1] = 1

        self.prob_map = prob_map
        return self.T1_map, self.T2_map, self.prob_map

    def phantom_display(self, title):
        plt.figure(figsize=(5, 5))

        plt.subplot(2, 2, 1)
        plt.imshow(self.T1_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} T1 Map")

        plt.subplot(2, 2, 2)
        plt.imshow(self.T2_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} T2 Map")

        plt.subplot(2, 2, 3)
        plt.imshow(self.prob_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} Probability Map")

        plt.tight_layout()
        plt.show()



## Phantom class (version 3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class VirtualPhantom():
    def __init__(self, img_size):
        self.img_size = img_size
        self.inner_size = int(0.25* img_size)
        self.mixed_size = int(0.50 * img_size)
        self.outer_size = int(0.75 * img_size)
        self.upper_square = int(0.078 * img_size)

        # gap between upper square and mixed square
        self.gap = max(1, int(0.02 * img_size))

        # Inner square (pure myelin) - centered
        self.top_inner = (self.img_size - self.inner_size) // 2
        self.bottom_inner = self.top_inner + self.inner_size

        # Mixed square (mixed signals) - centered
        self.top_mixed = (self.img_size - self.mixed_size) // 2
        self.bottom_mixed = self.top_mixed + self.mixed_size

        # Outer square (pure WM) - centered
        self.top_outer = (self.img_size - self.outer_size) // 2
        self.bottom_outer = self.top_outer + self.outer_size

        # Upper square: centered horizontally, above mixed square, with a gap
        self.x0 = self.img_size // 2 - self.upper_square // 2
        self.x1 = self.x0 + self.upper_square

        self.y1 = self.top_mixed - self.gap
        self.y0 = self.y1 - self.upper_square

        if self.y0 < 0:
            self.y0 = 0
            self.y1 = self.upper_square


    def create_artefact_ROI_region(self, width):
        artefact_ROI_region = np.zeros([self.img_size, self.img_size], dtype = np.float64)
        artefact_ROI_region[self.top_inner-width : self.top_inner+width, 0:self.top_outer] = 1
        artefact_ROI_region[self.bottom_inner-width : self.bottom_inner+width, 0:self.top_outer] = 1

        artefact_ROI_region[self.top_inner-width : self.top_inner+width, self.img_size - self.top_outer: self.img_size] = 1
        artefact_ROI_region[self.bottom_inner-width : self.bottom_inner+width, self.img_size - self.top_outer: self.img_size] = 1

        artefact_ROI_region[0:self.top_outer, self.top_inner-width : self.top_inner+width] = 1
        artefact_ROI_region[0:self.top_outer, self.bottom_inner-width : self.bottom_inner+width] = 1

        artefact_ROI_region[self.img_size - self.top_outer: self.img_size, self.top_inner-width : self.top_inner+width] = 1
        artefact_ROI_region[self.img_size - self.top_outer: self.img_size, self.bottom_inner-width : self.bottom_inner+width] = 1

        return artefact_ROI_region
    
    def create_background_ROI_region(self):
        background_ROI_region = np.zeros([self.img_size, self.img_size], dtype = np.float64)
        background_ROI_region[0 : self.top_outer, 0:self.top_outer] = 1
        background_ROI_region[self.img_size-self.top_outer : self.img_size , 0:self.top_outer] = 1
        background_ROI_region[0:self.top_outer, self.img_size-self.top_outer : self.img_size] = 1
        background_ROI_region[self.img_size-self.top_outer : self.img_size , self.img_size-self.top_outer : self.img_size] = 1
        return background_ROI_region


    def create_maps(self, t1, t2, inner, mixed, outer, upper_sqaure=True):
        # T1 map
        self.T1_map = np.full([self.img_size, self.img_size], t1, dtype=np.float64)

        # T2 map
        self.T2_map = np.full([self.img_size, self.img_size], t2, dtype=np.float64)

        # Probability map
        prob_map = np.zeros([self.img_size, self.img_size], dtype=np.float64)
        prob_map[self.top_outer:self.bottom_outer, self.top_outer:self.bottom_outer] = outer
        prob_map[self.top_mixed:self.bottom_mixed, self.top_mixed:self.bottom_mixed] = mixed
        prob_map[self.top_inner:self.bottom_inner, self.top_inner:self.bottom_inner] = inner
        if upper_sqaure:
            prob_map[self.y0:self.y1, self.x0:self.x1] = 1

        self.prob_map = prob_map
        return self.T1_map, self.T2_map, self.prob_map

    def phantom_display(self, title):
        plt.figure(figsize=(5, 5))

        plt.subplot(2, 2, 1)
        plt.imshow(self.T1_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} T1 Map")

        plt.subplot(2, 2, 2)
        plt.imshow(self.T2_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} T2 Map")

        plt.subplot(2, 2, 3)
        plt.imshow(self.prob_map, cmap='gray', vmin=0, vmax=1)
        plt.title(f"{title} Probability Map")

        plt.tight_layout()
        plt.show()

